In [ ]:
using LinearAlgebra, SparseArrays, Printf

# Build A, b, and the manufactured truth x⋆
function poisson1d_manufactured(n::Int)
    di = fill(2.0, n); dl = fill(-1.0, n-1); du = fill(-1.0, n-1)
    A  = spdiagm(-1 => dl, 0 => di, 1 => du)
    xg = range(1, n; step=1) ./ (n+1)                   # grid i/(n+1)
    xstar = @. sin(pi * xg)                             # "true" solution
    b = A * xstar                                       # manufactured RHS
    return A, b, xstar, collect(xg)
end

# Jacobi
function jacobi(A, b; x0=zeros(eltype(b), length(b)), tol=1e-10, maxit=20000)
    d = diag(A); Dinv = 1.0 ./ d
    R = A - spdiagm(0 => d)
    x = copy(x0); r = b - A*x; res = [norm(r)]
    for k in 1:maxit
        x .= Dinv .* (b - R*x)
        r .= b - A*x
        push!(res, norm(r))
        res[end] < tol && return x, res, k
    end
    return x, res, maxit
end

# Gauss–Seidel (forward sweep via triangular solve)
function gauss_seidel(A, b; x0=zeros(eltype(b), length(b)), tol=1e-10, maxit=20000)
    LHS = LowerTriangular(tril(A))
    U   = triu(A, 1)
    x = copy(x0); r = b - A*x; res = [norm(r)]
    tmp = similar(b)
    for k in 1:maxit
        mul!(tmp, U, x)
        x .= LHS \ (b - tmp)        # (D+L) x^{k+1} = b - U x^k
        r .= b - A*x
        push!(res, norm(r))
        res[end] < tol && return x, res, k
    end
    return x, res, maxit
end

# SOR
function sor(A, b; ω::Float64, x0=zeros(eltype(b), length(b)), tol=1e-10, maxit=20000)
    DωL = LowerTriangular(spdiagm(0 => diag(A)) + ω * tril(A, -1))
    U   = triu(A, 1)
    x = copy(x0); r = b - A*x; res = [norm(r)]
    tmp = similar(b)
    for k in 1:maxit
        mul!(tmp, U, x)
        x .= DωL \ (ω*b .- ω*tmp .+ (1-ω)*spdiagm(0 => diag(A))*x)
        r .= b - A*x
        push!(res, norm(r))
        res[end] < tol && return x, res, k
    end
    return x, res, maxit
end

# Plain CG (SPD)
function cg(A, b; x0=zeros(eltype(b), length(b)), tol=1e-10, maxit=10_000)
    x = copy(x0)
    r = b - A*x
    p = copy(r)
    rs = dot(r,r)
    res = [sqrt(rs)]
    for k in 1:maxit
        Ap = A*p
        α = rs / dot(p, Ap)
        @. x = x + α * p
        @. r = r - α * Ap
        rsnew = dot(r,r)
        push!(res, sqrt(rsnew))
        res[end] < tol && return x, res, k
        β = rsnew / rs
        @. p = r + β * p
        rs = rsnew
    end
    return x, res, maxit
end

# Demo / comparison
function demo(n::Int=200; tol=1e-10)
    A, b, xstar, _ = poisson1d_manufactured(n)

    # QR (SuiteSparse QR handles sparse A)
    F = qr(A)
    x_qr = F \ b
    r_qr = norm(b - A*x_qr); e_qr = norm(x_qr - xstar)/norm(xstar)

    x_j, rj, kj = jacobi(A,b; tol=tol)
    x_gs, rgs, kgs = gauss_seidel(A,b; tol=tol)
    ωopt = 2.0 / (1 + sin(pi/(n+1)))
    x_sor, rsor, ksor = sor(A,b; ω=ωopt, tol=tol)
    x_cg, rcg, kcg = cg(A,b; tol=tol)

    @printf("n=%d  (tol=%0.1e)\n", n, tol)
    @printf("QR:    residual=%0.3e  rel.error=%0.3e  iters= - \n", r_qr, e_qr)
    @printf("Jacobi:        residual=%0.3e  iters=%d\n", rj[end], kj)
    @printf("Gauss-Seidel:  residual=%0.3e  iters=%d\n", rgs[end], kgs)
    @printf("SOR(ω*=%.4f): residual=%0.3e  iters=%d\n", ωopt, rsor[end], ksor)
    @printf("CG:            residual=%0.3e  iters=%d\n", rcg[end], kcg)

    return (x_qr, x_j, x_gs, x_sor, x_cg), (r_qr, rj, rgs, rsor, rcg), xstar
end

# run it
demo();
